# Expresión génica en hepatocarcinoma (TCGA-LIHC)
### ¿Cambia la expresión de ciertos genes con el grado del tumor?

Cuaderno de la clase **VS Code desde cero** (Viernes de Bioinformática). Es el mismo análisis del script de la clase, pero en celdas: ejecutas una celda con **Shift+Enter** y ves su resultado justo debajo.

**Cómo usarlo**
1. Guarda este archivo en la **raíz de tu carpeta de proyecto** (por ejemplo `analisis-hcc/`). El cuaderno crea `data/` y `results/` junto a él.
2. Selecciona el kernel `Python (viernes-vscode)` (sección 0). Si todavía no lo tienes, créalo con la guía `primeros-pasos-conda/CREAR_KERNEL.md`.
3. **Regla del cuaderno: cada línea de código lleva su comentario** (lo que va después de `#`), para que se entienda qué hace sin buscar en otra parte.
4. Sigue de arriba hacia abajo. Si algo sale raro, **Restart** (arriba) y vuelve a correr todo desde el principio.

## 0. Seleccionar el kernel

Un cuaderno necesita un **kernel**: el Python que ejecuta las celdas. Antes de correr cualquier celda:

1. Arriba a la derecha, clic en **Select Kernel**.
2. Elige **Jupyter Kernel…** y después **Python (viernes-vscode)**. Si no aparece, pulsa el botón de refrescar de esa lista.
3. Si sigue sin aparecer, todavía no lo has creado: sigue la guía [CREAR_KERNEL.md](https://github.com/MiguelDiaz02/viernes-vscode/blob/main/primeros-pasos-conda/CREAR_KERNEL.md) (se hace una sola vez, desde la terminal) y vuelve aquí.

In [ ]:
import sys  # sys: información del Python que está corriendo este cuaderno

print("Python:", sys.version.split()[0])  # solo el número de versión, por ejemplo 3.12.7
print("Ejecutable:", sys.executable)  # ruta del Python que usa el kernel

La ruta de **Ejecutable** debe contener `viernes-vscode` (si usas conda) o `.venv` (si usas un entorno virtual). Si ves otra cosa, estás en otro Python: vuelve a **Select Kernel** y elige el correcto.

---
## 1. Bibliotecas

`import` trae una biblioteca. Los alias `pd`, `np` y `plt` son universales. `pandas` maneja tablas, `numpy` hace matemáticas y `matplotlib` dibuja. `pathlib` y `urllib` ya vienen con Python.

In [ ]:
from pathlib import Path  # Path: rutas de archivos que funcionan igual en Windows, Mac y Linux
from urllib.request import urlretrieve  # urlretrieve: baja un archivo de internet y lo guarda

import matplotlib.pyplot as plt  # matplotlib dibuja las figuras; plt es su alias universal
import numpy as np  # numpy hace matemáticas con arreglos; np es su alias universal
import pandas as pd  # pandas maneja tablas; pd es su alias universal

## 2. Rutas del proyecto y carpetas

Los nombres en MAYÚSCULAS son constantes. En un cuaderno no existe `__file__`, así que la raíz del proyecto es la carpeta de trabajo (`Path.cwd()`), que en VS Code es la carpeta donde guardaste el cuaderno. `mkdir` crea cada carpeta; `exist_ok=True` evita el error si ya existe.

> En un cuaderno también podrías escribir `!mkdir data`, pero esa orden cambia según el sistema (Windows, Mac, Linux). `mkdir` de Python funciona igual en los tres.

In [ ]:
RAIZ = Path.cwd()  # raíz del proyecto: la carpeta de trabajo, donde guardaste el cuaderno
CARPETA_DATOS = RAIZ / "data"  # "/" une rutas; aquí guardaremos los datos
CARPETA_TABLAS = RAIZ / "results" / "tablas"  # aquí irán las tablas de resultados
CARPETA_FIGURAS = RAIZ / "results" / "figuras"  # aquí irán las figuras

for carpeta in (CARPETA_DATOS, CARPETA_TABLAS, CARPETA_FIGURAS):  # repite lo de abajo para cada carpeta
    carpeta.mkdir(parents=True, exist_ok=True)  # la crea (con sus carpetas padre); si ya existe, no pasa nada

print("Proyecto en:", RAIZ)  # confirma dónde quedó el proyecto

## 3. Descargar los datos

Los datos vienen de TCGA (un consorcio público de cáncer), recortados a 12 genes y 371 tumores de hepatocarcinoma. La función `descargar` baja un archivo y, si ya está en `data/`, no lo vuelve a bajar. Fuente: *The Cancer Genome Atlas Research Network, Cell 2017; 169(7):1327-1341*.

In [ ]:
URL_BASE = "https://raw.githubusercontent.com/MiguelDiaz02/viernes-vscode/main/data"  # dirección de la carpeta con los datos
ARCHIVOS = ["hcc_expresion.csv", "hcc_clinicos.csv"]  # nombres de los dos archivos que necesitamos


def descargar(nombre):  # función: recibe el nombre de un archivo y lo baja a data/
    destino = CARPETA_DATOS / nombre  # dónde quedará el archivo en tu computadora
    if destino.exists():  # si ya lo bajaste antes...
        print("Ya existe:", destino.name)  # ...avisa
        return destino  # ...y termina sin bajarlo otra vez
    urlretrieve(f"{URL_BASE}/{nombre}", destino)  # baja el archivo (la f deja poner variables entre llaves)
    print("Descargado:", destino.name)  # confirma la descarga
    return destino  # entrega la ruta del archivo por si la queremos usar


for archivo in ARCHIVOS:  # para cada uno de los dos archivos...
    descargar(archivo)  # ...llama a la función

## 4. Cargar los datos

`read_csv` lee un archivo; `index_col` dice qué columna son los nombres de fila; `na_values` explica a pandas qué textos significan «vacío». `.shape` da (filas, columnas) y `.head()` muestra las primeras filas.

Resultado esperado: expresión de **12 genes × 371 muestras** y datos clínicos de **371 pacientes × 7 variables**.

In [ ]:
expresion = pd.read_csv(CARPETA_DATOS / "hcc_expresion.csv", index_col="gen")  # lee la tabla; la columna "gen" da los nombres de fila
clinicos = pd.read_csv(  # lee la tabla clínica con varias opciones:
    CARPETA_DATOS / "hcc_clinicos.csv",  # el archivo a leer
    index_col="id_paciente",  # la columna "id_paciente" da los nombres de fila
    na_values=["[Not Available]", "[Not Applicable]", "[Unknown]", "[Discrepancy]"],  # textos que significan "vacío"
)  # cierra la llamada a read_csv

print("Expresión (genes x muestras):", expresion.shape)  # (filas, columnas) de la tabla de expresión
print("Clínicos (pacientes x variables):", clinicos.shape)  # (filas, columnas) de la tabla clínica

In [ ]:
expresion.iloc[:, :5]  # primeras 5 columnas (muestras) de la tabla; la última línea de una celda se muestra sola

In [ ]:
clinicos.head()  # primeras 5 filas de la tabla clínica

## 5. Funciones

Una función es una receta con nombre; su nombre es un verbo porque hace algo.

- `log2_expresion` comprime la escala: pasa los valores a log2(valor + 1).
- `agrupar_grado` junta los grados del tumor en dos grupos: G1-G2 (bajo) y G3-G4 (alto).
- `resumir_por_grupo` calcula el promedio de cada gen por grupo y el cambio entre grupos (en log2).

In [ ]:
def log2_expresion(tabla):  # función: recibe una tabla de expresión
    return np.log2(tabla + 1)  # log2(valor + 1): comprime la escala y evita log2(0)


def agrupar_grado(grado):  # función: recibe el grado de un tumor (G1, G2, G3 o G4)
    if grado in ("G1", "G2"):  # si el grado es G1 o G2...
        return "G1-G2"  # ...pertenece al grupo de grado bajo
    return "G3-G4"  # en cualquier otro caso, al grupo de grado alto


def resumir_por_grupo(datos, genes):  # función: recibe la tabla de trabajo y la lista de genes
    promedios = datos.groupby("grupo_grado")[genes].mean().T  # promedio de cada gen por grupo; .T voltea la tabla
    promedios["cambio_log2"] = promedios["G3-G4"] - promedios["G1-G2"]  # columna nueva: diferencia entre grupos
    return promedios.sort_values("cambio_log2", ascending=False)  # ordena del mayor al menor cambio

## 6. Preparar la tabla de trabajo

Pasos: (1) log2 de la expresión; (2) `.T` voltea la tabla para que las muestras queden en filas; (3) el identificador de muestra `TCGA-2V-A95S-01` se recorta a los primeros 12 caracteres, `TCGA-2V-A95S`, que es el del paciente; (4) `.join` une las dos tablas por ese identificador; (5) se quitan los pacientes sin grado y (6) se agrega la columna del grupo.

Resultado esperado: una tabla de **366 filas**, con **232** pacientes en G1-G2 y **134** en G3-G4.

In [ ]:
genes = list(expresion.index)  # lista con los nombres de los 12 genes (los nombres de fila)

datos = log2_expresion(expresion).T  # aplica log2 y voltea: muestras en filas, genes en columnas
datos.index = datos.index.str[:12]  # recorta TCGA-2V-A95S-01 a TCGA-2V-A95S (id del paciente)
datos.index.name = "id_paciente"  # nombre del índice, igual que en la tabla clínica
datos = datos.join(clinicos)  # une las dos tablas por id_paciente
datos = datos.dropna(subset=["grado"])  # quita los pacientes sin grado
datos["grupo_grado"] = datos["grado"].map(agrupar_grado)  # columna nueva: aplica agrupar_grado a cada paciente

print("Tabla de trabajo:", datos.shape)  # esperado: (366, 20)
print(datos["grupo_grado"].value_counts())  # cuántos pacientes hay en cada grupo

## 7. Resumen por grupo

`cambio_log2` es la diferencia entre el promedio de G3-G4 y el de G1-G2: **positivo** significa más expresión en tumores de grado alto. Los genes de referencia (GAPDH, ACTB) casi no cambian, como se espera. La tabla se guarda en `results/tablas/`.

In [ ]:
resumen = resumir_por_grupo(datos, genes)  # llama a la función: promedios y cambio por gen
resumen.round(3).to_csv(CARPETA_TABLAS / "resumen_por_grado.csv")  # guarda la tabla con 3 decimales

resumen.round(2)  # muestra la tabla con 2 decimales

## 8. Figura

Un diagrama de caja por gen y grupo. Hay tendencia, pero también mucho traslape entre grupos: esto describe los datos, no es una prueba estadística. La figura se guarda en `results/figuras/`.

In [ ]:
genes_figura = ["AFP", "CA9", "TOP2A", "ALB"]  # los 4 genes que vamos a dibujar

fig, ejes = plt.subplots(1, len(genes_figura), figsize=(12, 3.5))  # una fila de 4 recuadros, de 12 x 3.5 pulgadas
for eje, gen in zip(ejes, genes_figura):  # recorre cada recuadro junto con su gen
    grado_bajo = datos.loc[datos["grupo_grado"] == "G1-G2", gen]  # expresión del gen en tumores de grado bajo
    grado_alto = datos.loc[datos["grupo_grado"] == "G3-G4", gen]  # expresión del gen en tumores de grado alto
    eje.boxplot([grado_bajo, grado_alto], tick_labels=["G1-G2", "G3-G4"])  # dos cajas, una por grupo
    eje.set_title(gen)  # título del recuadro: el nombre del gen

ejes[0].set_ylabel("log2(expresión + 1)")  # etiqueta del eje vertical, solo en el primer recuadro
fig.suptitle("Expresión por grado tumoral (TCGA-LIHC)")  # título general de la figura
fig.tight_layout()  # acomoda los recuadros para que no se encimen
fig.savefig(CARPETA_FIGURAS / "expresion_por_grado.png", dpi=150)  # guarda la figura como imagen
plt.show()  # la muestra aquí en el cuaderno

print("Figura guardada en", CARPETA_FIGURAS / "expresion_por_grado.png")  # confirma dónde quedó

---
**Lo que quedó en tu carpeta**

```
analisis-hcc/
├── analisis_hcc.ipynb
├── data/        hcc_expresion.csv, hcc_clinicos.csv
└── results/     tablas/resumen_por_grado.csv, figuras/expresion_por_grado.png
```